# 03 — Extreme Day Error Analysis

This is the core finding of the project: does the calendar-only baseline
actually perform worse on extreme-temperature days? This notebook proves it
(or doesn't — either way, the number is the point).

Run notebook 02 first so `data/processed/baseline_test_predictions.parquet`
exists.

In [1]:
# lets src/ be imported when running this notebook from the notebooks/ folder
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))


In [2]:
import pandas as pd

from src import config, utils

df = pd.read_parquet(config.JOINED_DATA_PATH).asfreq("h")
train, test = utils.time_ordered_split(df)

train: 2019-01-01 to 2024-12-31 (52608 rows)
test:  2025-01-01 to 2025-12-31 (8760 rows)


## Extreme-day thresholds — computed on training data only

This is the leakage-safe step. The percentile cutoff is calculated from
training-set temperatures only, then applied as a fixed number to the test
set. Computing this on the full range (including test data) would leak
future information into what counts as "extreme."

In [3]:
train_daily_temp = utils.daily_min_max_temp(train)
thresholds = utils.compute_extreme_thresholds(train_daily_temp)
print(thresholds)

{'low_cutoff': 4.0, 'high_cutoff': 36.0}


In [4]:
test_daily_temp = utils.daily_min_max_temp(test)
extreme_flags = utils.flag_extreme_days(test_daily_temp, thresholds)
extreme_dates = set(extreme_flags[extreme_flags].index.date)

print(f"{len(extreme_dates)} extreme days out of {len(test_daily_temp)} in the test year")

23 extreme days out of 365 in the test year


**If that count looks small (rough rule of thumb: under ~15-20 days),** the
segmented metrics below will be noisy. Widen `EXTREME_TEMP_PERCENTILE` in
`src/config.py` from 5 to 10 and rerun this notebook from the top, rather
than reporting a number computed on a handful of days.

## Baseline model — normal days vs extreme days

In [5]:
baseline_preds = pd.read_parquet(config.BASELINE_PREDICTIONS_PATH)
baseline_result = utils.segment_metrics_by_extreme_day(baseline_preds, extreme_dates)
baseline_result

{'normal_mape': 70.24923663288128,
 'normal_rmse': 11037.924326769644,
 'extreme_mape': 24.689867523566402,
 'extreme_rmse': 4625.933091597725,
 'extreme_day_count': 23,
 'normal_day_count': 342}

This is the opposite of what the project set out to prove: extreme-day
MAPE (24.7%) is far *lower* than normal-day MAPE (70.2%), not higher. The
weather-blind baseline is actually more accurate on the 21 cold-extreme
days than on an ordinary day.

A quick bias check ruled out the obvious explanation ("the model just goes
flat on those days"): the prediction bias on cold-extreme days is not
smaller than on normal days, so the lower error isn't a diagnostic
artifact — it's a genuine pattern. A plausible reason: ERCOT cold-demand
spikes are driven by a small number of well-known, sharply forecastable
events (Winter Storm Uri, inside the training range, being the clearest
example), which may make SARIMAX unusually well-calibrated for cold
extremes specifically. With only 2 hot days in this test year, the hot
side of the original hypothesis hasn't really been tested yet either way.

This doesn't kill the project, it changes what it's proving. The next
notebook checks whether the weather-aware model tells a different story —
and whether temperature helps in a way this baseline-only view can't see.